In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import bisect

from mff.new_cap import (
    calc_degressive_and_capping_steps,
    compute_capped_subsidies,
    compute_dabis_support_summary,
    find_cur_new_equal_root,
    find_flat_rate,
    generate_labels_from_bins,
    read_base_data,
)
from mff.plots import (
    PolicyParams,
    build_rate_area_matrix,
    plot_allocation_with_fixed_rate,
    plot_avg_change_vs_farmer_count_by_area_class,
    plot_avg_subsidy_per_ha_current_by_area_class,
    plot_diff_dual_axis,
    plot_diff_pct,
    plot_per_ha,
    plot_per_ha_support_comparison_by_area_class,
    plot_perc_diff_heatmap_by_rate_and_area_class,
    plot_reduction,
    plot_subsidy_rate_sweep_by_area_class,
    plot_subsidy_rate_sweep_overview,
    plot_support_summary_by_area_class,
    plot_total,
    plot_total_capped_loss_by_area_class,
    plot_yfs_distribution,
)
from mff.utils import c_round

In [ ]:
data = read_base_data(2024)

In [ ]:
# from data_tools.db import Manager

# import polars as pl
# from sqlalchemy import text, bindparam

# engine = Manager("mvh-admin", "mvh").engine

# def query_area(regszam_lst):
#     query = text("""
#         SELECT regszam,
#                sum(terulet_elf) as ter
#         FROM ek_adatok.tera
#         WHERE ev = 2024
#           AND terulet_elf > 0
#           AND tam_nem_ig = 0
#           AND teruletalapu_alaptamogatas = 1
#           AND regszam IN :regszam_lst
#         GROUP BY ev, regszam
#     """).bindparams(
#         bindparam("regszam_lst", expanding=True),
#     )

#     return pl.read_database(
#         query,
#         connection=engine,
#         execute_options={
#             "parameters": {
#                 "regszam_lst": regszam_lst,
#             }
#         },
#     ).to_pandas()


# data_merged = pd.read_excel("farms_to_merge.xlsx", engine="calamine")
# regszam_lst = data_merged["regszam"].astype("Int64").dropna().to_list()
# data_area = query_area(regszam_lst)

# data_merged = data_merged[["azonosito", "regszam", "nev", "cegnev", "cim"]].dropna().reset_index(drop=True)
# data_merged = data_merged.merge(data_area, how="left", on="regszam")
# data_merged = data_merged.dropna().reset_index(drop=True)
# data_merged.to_excel("data_merged_queried.xlsx", index=False)

# data_merged = pd.read_excel("data_merged_queried.xlsx", engine="calamine")
# data_merged["ossz_ter"] = data_merged.groupby("azonosito")["ter"].transform("sum")
# data_merged.to_excel("data_merged_queried.xlsx", index=False)

In [ ]:
data_merged = pd.read_excel("data_merged_queried.xlsx")

cols = [
    "area_biss_criss",
    "area_yfs",
    "area_yfs_cur_eligible",
    "subs_biss",
    "subs_redist",
    "subs_yfs",
]


df = data.merge(
    data_merged[["regszam", "azonosito"]],
    on="regszam",
    how="left",
)


to_merge = df[df["azonosito"].notna()]


rest = df[df["azonosito"].isna()]


merged = to_merge.groupby("azonosito", as_index=False)[cols].sum()


merged["regszam"] = to_merge.groupby("azonosito")["regszam"].first().values


out = pd.concat(
    [
        rest[data.columns],  # untouched
        merged[data.columns],  # collapsed
    ],
    ignore_index=True,
)

In [ ]:
from mff.new_cap import cal_redist_vec

out["area_yfs_cur_eligible"] = np.minimum(out["area_yfs"], 300)
out["subs_biss"] = out["area_biss_criss"] * 148.1
out["subs_redist"] = cal_redist_vec(out["area_biss_criss"])
out["subs_yfs"] = out["area_yfs_cur_eligible"] * 90

In [ ]:
out

In [ ]:
data_merged

In [ ]:
# BISS+CRISS+CIS-FY
BUDGET = 943_167_086.78
YFS_PER_HA = 90
REDIST_PER_HA = (80, 40)

In [ ]:
# YFS fajlagos (2024-ben)
18.6 * 1e6 / 199765.5708

In [ ]:
compute_dabis_support_summary(data)

In [ ]:
solution_with_yfs = find_flat_rate(data, BUDGET, YFS_PER_HA, REDIST_PER_HA)

DABIS_PER_HA = c_round(solution_with_yfs.root, 2)

result = pd.DataFrame(
    {
        "YFS per/ha": [YFS_PER_HA],
        "DABIS per/ha": [DABIS_PER_HA],
        "REDIST per/ha": [REDIST_PER_HA],
    }
)

print(result)

In [ ]:
calc_degressive_and_capping_steps(data, DABIS_PER_HA, REDIST_PER_HA, 90, "yf")

In [ ]:
calc_degressive_and_capping_steps(data, DABIS_PER_HA, REDIST_PER_HA, 0, "not-yf")

In [ ]:
calc_degressive_and_capping_steps(data, DABIS_PER_HA, REDIST_PER_HA, 90, "all")

In [ ]:
calc_degressive_and_capping_steps(data, DABIS_PER_HA, REDIST_PER_HA, 90)

In [ ]:
calc_degressive_and_capping_steps(
    data, DABIS_PER_HA, REDIST_PER_HA, 0, "not-yf"
).to_clipboard()

In [ ]:
policy_non_yfs = PolicyParams(
    base_payment_per_ha=DABIS_PER_HA, yfs_per_ha=0, redist_params=REDIST_PER_HA
)

policy_yfs = PolicyParams(
    base_payment_per_ha=DABIS_PER_HA, yfs_per_ha=90, redist_params=REDIST_PER_HA
)

In [ ]:
plot_per_ha(
    policy_non_yfs,
    label1="DABIS",
    label2="BISS+CRISS",
    title="Hektáronkénti támogatás alakulása (nem fiatal gazdálkodó)",
    output_path="output/abra_01_per_ha_non_yf.png",
)

In [ ]:
plot_per_ha(
    policy_yfs,
    label1="DABIS",
    label2="BISS+CRISS+CIS-YF",
    title="Hektáronkénti támogatás alakulása (fiatal gazdálkodó)",
    output_path="output/abra_02_per_ha_yf.png",
)

In [ ]:
plot_total(
    policy_non_yfs,
    label1="DABIS",
    label2="BISS+CRISS",
    title="Támogatás alakulása (nem fiatal gazdálkodó)",
    output_path="output/abra_03_total_non_yf.png",
)

In [ ]:
plot_total(
    policy_yfs,
    label1="DABIS",
    label2="BISS+CRISS+CIS-YF",
    title="Teljes támogatás alakulása (fiatal gazdálkodó)",
    output_path="output/abra_04_total_yf.png",
)

In [ ]:
plot_diff_pct(
    policy_non_yfs,
    "Hektáronkénti támogatás százalékos eltérése üzemméret szerint (nem fiatal gazdálkodó)",
    "DABIS és BISS+CRISS közötti eltérés",
    output_path="output/abra_05_diff_non_yf_ver01.png",
)

In [ ]:
plot_diff_dual_axis(
    policy_non_yfs,
    "Hektáronkénti támogatás százalékos eltérése és üzemszám eloszlása (nem fiatal gazdálkodó)",
    "DABIS és BISS+CRISS közötti eltérés",
    output_path="output/abra_05_diff_non_yf_ver02.png",
    farm_areas=data.loc[data["area_yfs"] == 0, "area_biss_criss"].tolist(),
    cum_mode="farm_number",
)

In [ ]:
plot_diff_dual_axis(
    policy_non_yfs,
    "Hektáronkénti támogatás százalékos eltérése és üzemszám eloszlása (nem fiatal gazdálkodó)",
    "DABIS és BISS+CRISS közötti eltérés",
    output_path="output/abra_05_diff_non_yf_ver03.png",
    farm_areas=data.loc[data["area_yfs"] == 0, "area_biss_criss"].tolist(),
    cum_mode="area",
)

In [ ]:
plot_diff_pct(
    policy_yfs,
    "Hektáronkénti támogatás százalékos eltérése üzemméret szerint (fiatal gazdálkodó)",
    "DABIS és BISS+CRISS+CIS-YF közötti eltérés",
    output_path="output/abra_06_diff_yf_ver01.png",
)

In [ ]:
plot_diff_dual_axis(
    policy_yfs,
    "Hektáronkénti támogatás százalékos eltérése és üzemszám eloszlása (fiatal gazdálkodó)",
    "DABIS és BISS+CRISS+CIS-YF közötti eltérés",
    output_path="output/abra_06_diff_non_yf_ver02.png",
    cum_mode="farm_number",
    farm_areas=data.loc[data["area_yfs"] > 0, "area_yfs"].tolist(),
)

In [ ]:
plot_diff_dual_axis(
    policy_yfs,
    "Hektáronkénti támogatás százalékos eltérése és üzemszám eloszlása (fiatal gazdálkodó)",
    "DABIS és BISS+CRISS+CIS-YF közötti eltérés",
    output_path="output/abra_06_diff_non_yf_ver03.png",
    cum_mode="area",
    farm_areas=data.loc[data["area_yfs"] > 0, "area_yfs"].tolist(),
)

In [ ]:
plot_reduction(
    policy_yfs,
    "Elvonás mértéke üzemméret függvényében (nem fiatal gazdálkodó)",
    "Degresszió és capping után megmaradó támogatás aránya (%)",
    output_path="output/non_yf_diff_02.png",
)

In [ ]:
plot_reduction(
    policy_non_yfs,
    "Elvonás mértéje üzemméret függvényében (nem fiatal gazdálkodó)",
    "A degresszió és capping Elvonási",
    output_path="output/non_yf_diff_02.png",
)

In [ ]:
plot_diff_pct(
    policy_yfs,
    "Százalékos különbség alakulása (fiatal gazdálkodó)",
    "DABIS",
    output_path="output/yf_dif_02.png",
)

In [ ]:
# todo non-YFS, lépcsők
# terület és állatállmány -> sertéstartók
# terület share alatta?
plot_yfs_distribution(data, 138.66)

In [ ]:
x_equal = find_cur_new_equal_root(DABIS_PER_HA, REDIST_PER_HA, 90)
res = c_round(x_equal, 2)

print(f"Az optimalizáció eredménye: {res} ha.")

## Fontos -> fenti ábra precizitását növelni kell
farm_size_threshold = res

In [ ]:
res = compute_capped_subsidies(data, 196, 120, REDIST_PER_HA)

total = res["subs_capped"].sum()
val = c_round(total, 2)

val_fmt = f"{val:,.0f}".replace(",", " ")

print(f"Fajlagos támogatás: 200 EUR, Szükséges keret: {val_fmt} EUR.")

In [ ]:
BISS_env = 735.3 * 1e6
YFS_env = 18.6 * 1e6
AOP_env = 202.125 * 1e6
REDIST_env = 189.1 * 1e6

In [ ]:
18.6 / 199

In [ ]:
YFS_env + BISS_env + AOP_env

### Direct Payments – allocated commitments 2021–2027 (EUR million, current prices)

| Member State (MS) | 2021     | 2022     | 2023     | 2024     | 2025     | 2026     | 2027     | Total 2021-2027 |
|-------------------|----------|----------|----------|----------|----------|----------|----------|-----------------|
| HU                | 1 305,72 | 1 305,72 | 1 347,40 | 1 347,40 | 1 347,40 | 1 347,40 | 1 243,19 | 9 244,22        |

In [ ]:
def calc_alloc_based(val):
    data_with_subs = compute_capped_subsidies(data, val, 0, REDIST_PER_HA)
    return data_with_subs["subs_capped"].sum() / 1e6


def calc_subs_per_acre(val):
    data_with_subs = compute_capped_subsidies(data, val, 0, REDIST_PER_HA)
    return data_with_subs["subs_capped"].sum() / data_with_subs["area_biss_criss"].sum()


def target_func(val, target):
    return calc_subs_per_acre(val) - target


lower_bound = 100
upper_bound = 500

x_min = bisect(target_func, lower_bound, upper_bound, args=(130,))

x_max = bisect(target_func, lower_bound, upper_bound, args=(240,))

x_min, x_max

In [ ]:
# Define interval
values = np.linspace(x_min, x_max, 100)

# Calculate results
alloc_results = [calc_alloc_based(v) for v in values]
subs_per_acre_results = [calc_subs_per_acre(v) for v in values]

In [ ]:
plot_allocation_with_fixed_rate(240, subs_per_acre_results, alloc_results, values)

In [ ]:
# ALLOCATION = 612 * 1e6
ALLOCATION = 0.75 * 924.4 * 1e6
# ALLOCATION = 924.4 * 1e6
ALLOCATION = 916.55 * 1e6

bins = [0, 10, 150, 300, 1200, float("inf")]
labels = generate_labels_from_bins(bins)

dabis_per_ha = c_round(find_flat_rate(data, ALLOCATION, 0, REDIST_PER_HA).root, 2)
print(f"Az optimalizáció eredménye: {dabis_per_ha} EUR/ha.")

plot_per_ha_support_comparison_by_area_class(
    data, DABIS_PER_HA, REDIST_PER_HA, bins, labels, ALLOCATION
)

In [ ]:
plot_support_summary_by_area_class(
    data, DABIS_PER_HA, REDIST_PER_HA, bins, labels, ALLOCATION
)

In [ ]:
plot_avg_change_vs_farmer_count_by_area_class(
    data, DABIS_PER_HA, REDIST_PER_HA, bins, labels
)

In [ ]:
plot_total_capped_loss_by_area_class(data, DABIS_PER_HA, REDIST_PER_HA, bins, labels)

In [ ]:
plot_avg_subsidy_per_ha_current_by_area_class(
    data, DABIS_PER_HA, REDIST_PER_HA, bins, labels
)

In [ ]:
subsidy_rates = np.linspace(100, 300, 50)
mtx = build_rate_area_matrix(data, REDIST_PER_HA, bins, labels, subsidy_rates)
mtx["area_class"] = pd.Categorical(mtx["area_class"], categories=labels, ordered=True)

In [ ]:
plot_subsidy_rate_sweep_by_area_class(mtx)

In [ ]:
plot_subsidy_rate_sweep_overview(mtx)

In [ ]:
plot_perc_diff_heatmap_by_rate_and_area_class(mtx)